## DB

In [1]:
# [DB 연결 1] postgresql 연결 라이브러리 설치
%pip install psycopg2-binary

In [2]:
# [DB 연결 2] 라이브러리
import getpass
import threading
import time
import io
import psycopg2         # PostgreSQL 접속
import re


import pandas as pd
import numpy as np

In [3]:
host   = "YOUR_NGROK_HOST"    # ← ngrok에 표시된 호스트 (tcp:// 제외)
port   = 0                   # ← ngrok에 표시된 포트
dbname = "YOUR_DB_NAME"
user   = "YOUR_DB_USER"
pw     = getpass.getpass("비밀번호: ")



def _connect():
    return psycopg2.connect(
        host=host, port=port, dbname=dbname, user=user, password=pw,
        connect_timeout=15,
        keepalives=1, keepalives_idle=30,          # 연결 유지용 TCP keepalive
        keepalives_interval=10, keepalives_count=5,
    )

conn = _connect()


def ensure_conn():
    """연결이 끊겼으면 재접속 (청크 사이에 호출)"""
    global conn
    try:
        if conn.closed:
            raise psycopg2.InterfaceError("closed")
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
        conn.rollback()                             # 확인용 트랜잭션 정리
    except Exception:
        try:
            conn.close()
        except Exception:
            pass
        conn = _connect()
        print("[재접속 완료]")

with conn.cursor() as cur:
    cur.execute("SELECT current_user, version()")
    print("접속 성공:", cur.fetchone())
conn.rollback()

비밀번호: ··········
접속 성공: ('YOUR_DB_USER', 'PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit')


# 공통함수 _sql

In [4]:
# ============================================================
# [DB 조회] SQL 결과를 DataFrame으로 변환하는 함수
#   - 컬럼명은 PostgreSQL 규칙대로 소문자로 반환됨 (예: rental_date)
#   - %s 는 파라미터 자리 (SQL 인젝션 방지, 값은 params 리스트로 전달)
# ============================================================
def query_df(sql, params=None):
    """SQL 결과 전체를 DataFrame으로 반환 (결과가 작을 때 사용)"""
    ensure_conn()                                   # 연결이 끊겼으면 재접속
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
    finally:
        conn.rollback()                             # 읽기 전용이므로 트랜잭션 정리
    return pd.DataFrame(rows, columns=cols)


def iter_query(sql, params=None, chunk_rows=500_000):
    """SQL 결과를 chunk_rows 행씩 DataFrame으로 반환하는 제너레이터 (대용량 처리용)
    서버사이드 커서를 써서 DB가 결과를 조금씩 보내므로 메모리를 거의 쓰지 않음"""
    ensure_conn()
    cur = conn.cursor(name="stream_cur")            # name 지정 = 서버사이드 커서
    cols = None
    try:
        cur.execute(sql, params)
        while True:
            rows = cur.fetchmany(chunk_rows)
            if not rows:
                break
            if cols is None:                        # 첫 fetch 이후에 컬럼 정보가 채워짐
                cols = [d[0] for d in cur.description]
            yield pd.DataFrame(rows, columns=cols)
    finally:
        cur.close()
        conn.rollback()

In [5]:
# ============================================================
# [DB 저장 공통함수 - 전처리 결과용] DataFrame 타입을 보고 DB 컬럼 타입을 자동 지정
#   - 기존 save_df(원본 보관용)는 전부 TEXT 로 저장하므로, 전처리 결과에는 이 함수를 사용
#   - if_exists: "fail"(기본, 이미 있으면 오류) / "append"(추가) / "replace"(삭제 후 다시 생성)
#   - pk: 기본키로 지정할 컬럼명(선택). 중복 값이 있으면 저장이 취소됨
#   - 빈 값(NA)과 빈 문자열은 모두 NULL 로 저장됨
# ============================================================
def _pg_type(s):
    """pandas 컬럼(Series) → PostgreSQL 타입"""
    if pd.api.types.is_bool_dtype(s):
        return "BOOLEAN"
    if pd.api.types.is_integer_dtype(s):
        mx = s.dropna().abs().max()
        return "INTEGER" if (pd.isna(mx) or mx < 2**31) else "BIGINT"
    if pd.api.types.is_float_dtype(s):
        return "DOUBLE PRECISION"
    if pd.api.types.is_datetime64_any_dtype(s):
        return "TIMESTAMP"
    return "TEXT"


def save_df_typed(df, table_name, conn, if_exists="fail", pk=None):
    if if_exists not in ("fail", "append", "replace"):
        raise ValueError('if_exists는 "fail", "append", "replace" 중 하나여야 합니다.')
    if df.empty:
        raise ValueError("저장할 데이터가 없습니다.")

    table_name = table_name.lower()
    columns = [str(c).lower() for c in df.columns]
    for name in [table_name, *columns]:
        if not re.fullmatch(r"[a-z][a-z0-9_]{0,62}", name):
            raise ValueError(f"사용할 수 없는 이름: {name}")
    if len(columns) != len(set(columns)):
        raise ValueError("중복된 컬럼명이 있습니다.")
    if pk and pk.lower() not in columns:
        raise ValueError(f"기본키 컬럼이 없습니다: {pk}")

    # 컬럼 정의: 컬럼명 + 자동 판별한 타입 (+ 기본키)
    col_defs = [f"{c} {_pg_type(df[orig])}" for c, orig in zip(columns, df.columns)]
    if pk:
        col_defs.append(f"PRIMARY KEY ({pk.lower()})")

    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)

    with conn.cursor() as cur:
        try:
            cur.execute("SELECT to_regclass(%s::text)", [table_name])
            exists = cur.fetchone()[0] is not None

            if exists and if_exists == "fail":
                raise ValueError(f"{table_name}: 이미 존재하는 테이블입니다. (if_exists로 append/replace 지정)")
            if exists and if_exists == "replace":
                cur.execute(f"DROP TABLE {table_name}")        # 트랜잭션 안이라 실패하면 원복됨
                exists = False

            if exists:   # append: 기존 테이블에 없는 컬럼이 있으면 중단
                cur.execute("""SELECT column_name FROM information_schema.columns
                               WHERE table_schema = current_schema() AND table_name = %s""", [table_name])
                # missing = [c for c in columns if c not in {r[0] for r in cur.fetchall()}]
                # 수정 _if_exists="append" 미동작
                existing = {r[0] for r in cur.fetchall()}
                missing = [c for c in columns if c not in existing]

                if missing:
                    raise ValueError(f"{table_name}에 없는 컬럼: {missing}")
            else:
                cur.execute(f"CREATE TABLE {table_name} ({', '.join(col_defs)})")

            cur.copy_expert(f"COPY {table_name} ({', '.join(columns)}) FROM STDIN WITH (FORMAT csv)", buf)
            conn.commit()
        except Exception:
            try:
                conn.rollback()
            except Exception:
                pass
            raise

    print(f"{table_name.upper()}: {len(df):,}행 저장 완료")

#1\. 따릉이 대여소 관련 전처리

##1-1. 따릉이 대여소

In [ ]:
# 값의 형식(날짜 문자열 모양, 성별 값 등)을 확인해야 전처리 방향을 정할 수 있음
df_sample = query_df("SELECT * FROM raw_bike_station LIMIT 10")
display(df_sample)
print(df_sample.dtypes)

,sta_loc,rent_id,rent_no,rent_nm,rent_id_nm,hold_num,sta_add1,sta_add2,sta_lat,sta_long,start_index,end_index,rnum
0,마포구,ST-10,00108,서교동 사거리,108. 서교동 사거리,12,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725,0,0,1
1,광진구,ST-100,00503,더샵스타시티 C동 앞,503. 더샵스타시티 C동 앞,15,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314,0,0,2
2,양천구,ST-1000,00729,서부식자재마트 건너편,729. 서부식자재마트 건너편,10,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840,0,0,3
3,양천구,ST-1002,00731,서울시 도로환경관리센터,731. 서울시 도로환경관리센터,10,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114,0,0,4
4,양천구,ST-1003,00732,신월중학교,732. 신월중학교,10,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048,0,0,5
5,양천구,ST-1004,00733,신정이펜하우스314동,733. 신정이펜하우스314동,10,서울특별시 양천구 신정이펜1로50,신정이펜하우스314동,37.51409912,126.83100128,0,0,6
6,양천구,ST-1005,00734,신트리공원 입구,734. 신트리공원 입구,10,서울특별시 양천구 신정동 310-8,신트리공원 입구,37.51395035,126.85605621,0,0,7
7,양천구,ST-1006,00735,영도초등학교,735. 영도초등학교,10,서울특별시 양천구 목동중앙로 70 서울영도초등학교,None,37.53637695,126.87151337,0,0,8
8,양천구,ST-1007,00736,오솔길공원,736. 오솔길공원,10,서울특별시 양천구 남부순환로70길11-9,오솔길공원,37.52219009,126.83670044,0,0,9
9,양천구,ST-1008,00737,장수공원,737. 장수공원,20,서울특별시 양천구 신정동 1235,None,37.52231216,126.84904480,0,0,10


sta_loc        object
rent_id        object
rent_no        object
rent_nm        object
rent_id_nm     object
hold_num       object
sta_add1       object
sta_add2       object
sta_lat        object
sta_long       object
start_index    object
end_index      object
rnum           object
dtype: object


In [ ]:
# ============================================================
# [정류장] raw_bike_station에서 필요한 컬럼만 조회
#   앞서 만든 query_df 함수 사용 (DB 컬럼명은 소문자)
# ============================================================
df_station = query_df("""
    SELECT sta_loc,      -- 행정구
           rent_id,      -- 정류장 아이디
           rent_no,      -- 정류장 번호
           rent_nm,      -- 정류장 이름
           sta_add1,     -- 주소1
           sta_add2,     -- 주소2
           sta_lat,      -- 위도
           sta_long      -- 경도
    FROM raw_bike_station
""")

print(df_station.shape)
print(df_station.dtypes)         # 전부 object(문자열)로 나옴
display(df_station.head())

(3244, 8)
sta_loc     object
rent_id     object
rent_no     object
rent_nm     object
sta_add1    object
sta_add2    object
sta_lat     object
sta_long    object
dtype: object


,sta_loc,rent_id,rent_no,rent_nm,sta_add1,sta_add2,sta_lat,sta_long
0,마포구,ST-10,00108,서교동 사거리,서울특별시 마포구 양화로 93,427,37.55274582,126.91861725
1,광진구,ST-100,00503,더샵스타시티 C동 앞,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.53666687,127.07359314
2,양천구,ST-1000,00729,서부식자재마트 건너편,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51037979,126.86679840
3,양천구,ST-1002,00731,서울시 도로환경관리센터,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.52989960,126.87654114
4,양천구,ST-1003,00732,신월중학교,서울특별시 양천구 화곡로 59,신월동 이마트,37.53955078,126.82830048


In [ ]:
# ============================================================
# [정류장 전처리] 원본(df_station)은 그대로 두고 복사본(df_station2)에서 변환
#   - rent_no            : 정수(int)   예) '00108' → 108
#   - sta_lat, sta_long  : 실수(float)
#   - 나머지             : 문자열(앞뒤 공백 제거)
#   - sta_add1 + sta_add2 → sta_addr (주소2가 비어 있으면 주소1만 사용)
# ============================================================
df_station2 = df_station.copy()

# 1) 문자열 컬럼: 문자열 타입으로 바꾸고 앞뒤 공백 제거
str_cols = ["sta_loc", "rent_id", "rent_nm", "sta_add1", "sta_add2"]
for c in str_cols:
    df_station2[c] = df_station2[c].astype("string").str.strip()

# 2) 정류장 번호: 정수로 변환 (앞자리 0 제거). 변환 불가 값은 결측(<NA>)이 됨
df_station2["rent_no"] = pd.to_numeric(df_station2["rent_no"], errors="coerce").astype("Int64")

# 3) 위도/경도: 실수로 변환. 변환 불가 값은 NaN
df_station2["sta_lat"]  = pd.to_numeric(df_station2["sta_lat"],  errors="coerce").astype("float64")
df_station2["sta_long"] = pd.to_numeric(df_station2["sta_long"], errors="coerce").astype("float64")

# 4) 주소 합치기: 주소2가 없으면(NULL 또는 빈 문자열) 주소1만, 있으면 "주소1 주소2"
a1 = df_station2["sta_add1"].fillna("")
a2 = df_station2["sta_add2"].fillna("")
df_station2["sta_addr"] = a1.where(a2 == "", a1 + " , " + a2).astype("string")

# 5) 합치기 전 주소 컬럼 제거 (남기려면 아래 줄을 주석 처리)
df_station2 = df_station2.drop(columns=["sta_add1", "sta_add2"])

# 6) 컬럼 순서 정리
df_station2 = df_station2[["sta_loc", "rent_id", "rent_no", "rent_nm", "sta_addr", "sta_lat", "sta_long"]]

# ---------------- 변환 결과 점검 ----------------
print(df_station2.dtypes, "\n")
print("결측 수:\n", df_station2.isna().sum(), "\n")
print("rent_no 변환 실패:", int(df_station2["rent_no"].isna().sum()), "건")
print("좌표가 0이거나 결측:", int(((df_station2["sta_lat"] == 0) | (df_station2["sta_long"] == 0)
                               | df_station2["sta_lat"].isna() | df_station2["sta_long"].isna()).sum()), "건")
print("rent_id 중복:", int(df_station2["rent_id"].duplicated().sum()), "건")
display(df_station2.head(10))

sta_loc     string[python]
rent_id     string[python]
rent_no              Int64
rent_nm     string[python]
sta_addr    string[python]
sta_lat            float64
sta_long           float64
dtype: object 

결측 수:
 sta_loc     0
rent_id     0
rent_no     0
rent_nm     0
sta_addr    0
sta_lat     0
sta_long    0
dtype: int64 

rent_no 변환 실패: 0 건
좌표가 0이거나 결측: 3 건
rent_id 중복: 0 건


,sta_loc,rent_id,rent_no,rent_nm,sta_addr,sta_lat,sta_long
0,마포구,ST-10,108,서교동 사거리,"서울특별시 마포구 양화로 93 , 427",37.552746,126.918617
1,광진구,ST-100,503,더샵스타시티 C동 앞,"서울특별시 광진구 아차산로 262 , 더샵스타시티 C동 앞",37.536667,127.073593
2,양천구,ST-1000,729,서부식자재마트 건너편,"서울특별시 양천구 신정동 236 , 서부식자재마트 건너편",37.510380,126.866798
3,양천구,ST-1002,731,서울시 도로환경관리센터,"서울특별시 양천구 목동동로 316-6 , 서울시 도로환경관리센터",37.529900,126.876541
4,양천구,ST-1003,732,신월중학교,"서울특별시 양천구 화곡로 59 , 신월동 이마트",37.539551,126.828300
5,양천구,ST-1004,733,신정이펜하우스314동,"서울특별시 양천구 신정이펜1로50 , 신정이펜하우스314동",37.514099,126.831001
6,양천구,ST-1005,734,신트리공원 입구,"서울특별시 양천구 신정동 310-8 , 신트리공원 입구",37.513950,126.856056
7,양천구,ST-1006,735,영도초등학교,서울특별시 양천구 목동중앙로 70 서울영도초등학교,37.536377,126.871513
8,양천구,ST-1007,736,오솔길공원,"서울특별시 양천구 남부순환로70길11-9 , 오솔길공원",37.522190,126.836700
9,양천구,ST-1008,737,장수공원,서울특별시 양천구 신정동 1235,37.522312,126.849045


In [ ]:
# 값의 형식(날짜 문자열 모양, 성별 값 등)을 확인해야 전처리 방향을 정할 수 있음
df_sample = query_df("SELECT * FROM raw_bike_station_master LIMIT 10")
display(df_sample)
print(df_sample.dtypes)

,rntls_id,addr1,addr2,lat,lot
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51038,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.0,0.0
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.5299,126.876541
5,ST-1003,서울특별시 양천구 화곡로 59,신월동 이마트,37.539551,126.8283
6,ST-1004,서울특별시 양천구 신정이펜1로50,신정이펜하우스314동,37.514099,126.831001
7,ST-1005,서울특별시 양천구 신정동 310-8,신트리공원 입구,37.51395,126.856056
8,ST-1006,서울특별시 양천구 목동중앙로 70 서울영도초등학교,None,37.536377,126.871513
9,ST-1007,서울특별시 양천구 남부순환로70길11-9,오솔길공원,37.52219,126.8367


rntls_id    object
addr1       object
addr2       object
lat         object
lot         object
dtype: object


In [ ]:
# ============================================================
# [정류장 마스터(교통정보시스템)] raw_bike_station_master 전체 컬럼 조회
# ============================================================
df_master = query_df("""
    SELECT rntls_id,   -- 정류장 아이디
           addr1,      -- 주소1
           addr2,      -- 주소2
           lat,        -- 위도
           lot         -- 경도
    FROM raw_bike_station_master
""")

print(df_master.shape)
print(df_master.dtypes)          # 전부 object(문자열)
display(df_master.head())

(3432, 5)
rntls_id    object
addr1       object
addr2       object
lat         object
lot         object
dtype: object


,rntls_id,addr1,addr2,lat,lot
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.51038,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.0,0.0
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.5299,126.876541


In [ ]:
# ============================================================
# [마스터 전처리] 원본(df_master)은 그대로 두고 복사본(df_master2)에서 변환
#   - lat, lot        : 실수(float)
#   - 나머지          : 문자열(앞뒤 공백 제거)
#   - addr1 + addr2   → addr (주소2가 비어 있으면 주소1만 사용)
# ============================================================
df_master2 = df_master.copy()

# 1) 문자열 컬럼: 문자열 타입으로 바꾸고 앞뒤 공백 제거
for c in ["rntls_id", "addr1", "addr2"]:
    df_master2[c] = df_master2[c].astype("string").str.strip()

# 2) 위도/경도: 실수로 변환 (변환 불가 값은 NaN)
df_master2["lat"] = pd.to_numeric(df_master2["lat"], errors="coerce").astype("float64")
df_master2["lot"] = pd.to_numeric(df_master2["lot"], errors="coerce").astype("float64")

# 3) 주소 합치기: 주소2가 없으면(NULL 또는 빈 문자열) 주소1만, 있으면 "주소1 주소2"
a1 = df_master2["addr1"].fillna("")
a2 = df_master2["addr2"].fillna("")
df_master2["addr"] = a1.where(a2 == "", a1 + " , " + a2).astype("string")

# 4) 합치기 전 주소 컬럼 제거, 컬럼 순서 정리
df_master2 = df_master2[["rntls_id", "addr", "lat", "lot"]]

# ---------------- 변환 결과 점검 ----------------
print(df_master2.dtypes, "\n")
print("결측 수:\n", df_master2.isna().sum(), "\n")
print("좌표가 0이거나 결측:", int(((df_master2["lat"] == 0) | (df_master2["lot"] == 0)
                               | df_master2["lat"].isna() | df_master2["lot"].isna()).sum()), "건")
print("rntls_id 중복:", int(df_master2["rntls_id"].duplicated().sum()), "건")
display(df_master2.head(10))

rntls_id    string[python]
addr        string[python]
lat                float64
lot                float64
dtype: object 

결측 수:
 rntls_id    0
addr        0
lat         0
lot         0
dtype: int64 

좌표가 0이거나 결측: 77 건
rntls_id 중복: 0 건


,rntls_id,addr,lat,lot
0,ST-10,"서울특별시 마포구 양화로 93 , 427",37.552746,126.918617
1,ST-100,"서울특별시 광진구 아차산로 262 , 더샵스타시티 C동 앞",37.536667,127.073593
2,ST-1000,"서울특별시 양천구 신정동 236 , 서부식자재마트 건너편",37.510380,126.866798
3,ST-1001,"서울특별시 양천구 남부순환로4길20 , 서서울호수공원",0.000000,0.000000
4,ST-1002,"서울특별시 양천구 목동동로 316-6 , 서울시 도로환경관리센터",37.529900,126.876541
5,ST-1003,"서울특별시 양천구 화곡로 59 , 신월동 이마트",37.539551,126.828300
6,ST-1004,"서울특별시 양천구 신정이펜1로50 , 신정이펜하우스314동",37.514099,126.831001
7,ST-1005,"서울특별시 양천구 신정동 310-8 , 신트리공원 입구",37.513950,126.856056
8,ST-1006,서울특별시 양천구 목동중앙로 70 서울영도초등학교,37.536377,126.871513
9,ST-1007,"서울특별시 양천구 남부순환로70길11-9 , 오솔길공원",37.522190,126.836700


In [ ]:
# ============================================================
# [정류장 비교] df_station2(따릉이) vs df_master2(교통정보시스템), 키 = 정류장 아이디
#   df_both : 양쪽에 모두 있는 정류장 (두 출처의 주소·좌표를 나란히 비교)
#   df_diff : 한쪽에만 있는 정류장 (data_source 컬럼으로 출처 구분)
# ============================================================
# 마스터 컬럼명을 따릉이와 맞추고, 비교용으로 _master 접미사를 붙임
m = df_master2.rename(columns={"rntls_id": "rent_id", "addr": "sta_addr",
                               "lat": "sta_lat", "lot": "sta_long"})

ids_bike   = set(df_station2["rent_id"].dropna())
ids_master = set(m["rent_id"].dropna())

# 1) 교집합: 양쪽에 모두 있는 정류장 (컬럼 접미사 _master = 교통정보시스템 값)
df_both = df_station2.merge(
    m.rename(columns={"sta_addr": "sta_addr_master",
                      "sta_lat": "sta_lat_master",
                      "sta_long": "sta_long_master"}),
    on="rent_id", how="inner"
)

# 2) 한쪽에만 있는 정류장 (isin으로 필터링 → 중복 행이 있어도 늘어나지 않음)
bike_only   = df_station2[~df_station2["rent_id"].isin(ids_master)].copy()
master_only = m[~m["rent_id"].isin(ids_bike)].copy()

# 3) 출처 컬럼 추가 (string 타입)
bike_only["data_source"]   = pd.Series("따릉이",        index=bike_only.index,   dtype="string")
master_only["data_source"] = pd.Series("교통정보시스템", index=master_only.index, dtype="string")

# 4) 두 결과를 하나로 합침 (마스터에만 있는 행은 sta_loc, rent_no, rent_nm 이 결측)
df_diff = pd.concat([bike_only, master_only], ignore_index=True)
df_diff["data_source"] = df_diff["data_source"].astype("string")

# ---------------- 결과 점검 ----------------
print(f"따릉이 정류장 아이디(고유): {len(ids_bike):,}개")
print(f"교통정보시스템 아이디(고유): {len(ids_master):,}개")
print(f"양쪽 모두 있음(df_both)          : {len(df_both):,}행")
print(f"따릉이에만 있음                   : {len(bike_only):,}행")
print(f"교통정보시스템에만 있음           : {len(master_only):,}행")
print(f"df_diff 합계                      : {len(df_diff):,}행")
print("\n" + str(df_diff["data_source"].value_counts()))
print("\ndf_diff dtypes:\n", df_diff.dtypes)
display(df_both.head())
display(df_diff.head())

따릉이 정류장 아이디(고유): 3,244개
교통정보시스템 아이디(고유): 3,432개
양쪽 모두 있음(df_both)          : 3,244행
따릉이에만 있음                   : 0행
교통정보시스템에만 있음           : 188행
df_diff 합계                      : 188행

data_source
교통정보시스템    188
Name: count, dtype: Int64

df_diff dtypes:
 sta_loc        string[python]
rent_id        string[python]
rent_no                 Int64
rent_nm        string[python]
sta_addr       string[python]
sta_lat               float64
sta_long              float64
data_source    string[python]
dtype: object


,sta_loc,rent_id,rent_no,rent_nm,sta_addr,sta_lat,sta_long,sta_addr_master,sta_lat_master,sta_long_master
0,마포구,ST-10,108,서교동 사거리,"서울특별시 마포구 양화로 93 , 427",37.552746,126.918617,"서울특별시 마포구 양화로 93 , 427",37.552746,126.918617
1,광진구,ST-100,503,더샵스타시티 C동 앞,"서울특별시 광진구 아차산로 262 , 더샵스타시티 C동 앞",37.536667,127.073593,"서울특별시 광진구 아차산로 262 , 더샵스타시티 C동 앞",37.536667,127.073593
2,양천구,ST-1000,729,서부식자재마트 건너편,"서울특별시 양천구 신정동 236 , 서부식자재마트 건너편",37.510380,126.866798,"서울특별시 양천구 신정동 236 , 서부식자재마트 건너편",37.510380,126.866798
3,양천구,ST-1002,731,서울시 도로환경관리센터,"서울특별시 양천구 목동동로 316-6 , 서울시 도로환경관리센터",37.529900,126.876541,"서울특별시 양천구 목동동로 316-6 , 서울시 도로환경관리센터",37.529900,126.876541
4,양천구,ST-1003,732,신월중학교,"서울특별시 양천구 화곡로 59 , 신월동 이마트",37.539551,126.828300,"서울특별시 양천구 화곡로 59 , 신월동 이마트",37.539551,126.828300


,sta_loc,rent_id,rent_no,rent_nm,sta_addr,sta_lat,sta_long,data_source
0,<NA>,ST-1001,<NA>,<NA>,"서울특별시 양천구 남부순환로4길20 , 서서울호수공원",0.0,0.0,교통정보시스템
1,<NA>,ST-1009,<NA>,<NA>,"서울특별시 양천구 신월동 263 , 신월 청소년 문화센터",0.0,0.0,교통정보시스템
2,<NA>,ST-1022,<NA>,<NA>,"서울특별시 은평구 진관동 2-6 , 입곡삼거리",0.0,0.0,교통정보시스템
3,<NA>,ST-1030,<NA>,<NA>,"서울특별시 은평구 녹번동 141-92 , 유성빌딩 옆",0.0,0.0,교통정보시스템
4,<NA>,ST-1060,<NA>,<NA>,"서울특별시 강동구 둔촌동 산 125 , 일자산 체육관",0.0,0.0,교통정보시스템


In [ ]:
# ============================================================
# [정류장 비교] df_both에서 주소·위도·경도 비교
#   - 모두 동일  : df_mismatch 생성 없이, df_both를 7개 컬럼만 남김
#   - 다른 행 있음: 먼저 df_mismatch에 담고(양쪽 값 포함), df_both는 7개 컬럼만 남김
# ============================================================
import numpy as np

COORD_TOL = 1e-5                     # 좌표 허용 오차(도). 약 1.1m. 0이면 완전히 같을 때만 동일
REMOVE_MISMATCH_FROM_BOTH = True     # True: 불일치 행은 df_mismatch로 옮기고 df_both에서는 제외
                                     # False: df_both에는 모든 행을 7개 컬럼으로 남김
keep_cols = ["sta_loc", "rent_id", "rent_no", "rent_nm", "sta_addr", "sta_lat", "sta_long"]

# 이미 정리된 df_both(비교용 *_master 컬럼 없음)로 다시 실행하면 비교할 수 없음
if "sta_addr_master" not in df_both.columns:
    raise RuntimeError("df_both가 이미 7개 컬럼으로 정리된 상태입니다. "
                       "교집합 셀(df_both 생성)부터 다시 실행하세요.")

# 1) 비교 함수
def norm_addr(s):
    """연속 공백을 하나로 줄이고 앞뒤 공백 제거 (둘 다 비어 있으면 동일로 판단)"""
    return s.fillna("").str.replace(r"\s+", " ", regex=True).str.strip()

def same_coord(a, b):
    """허용 오차(COORD_TOL) 이내면 동일, 둘 다 결측이면 동일"""
    a = a.to_numpy(dtype="float64", na_value=np.nan)
    b = b.to_numpy(dtype="float64", na_value=np.nan)
    return np.isclose(a, b, rtol=0, atol=COORD_TOL) | (np.isnan(a) & np.isnan(b))

# 2) 주소 / 위도 / 경도 비교 (True = 동일)
same_addr = (norm_addr(df_both["sta_addr"]) == norm_addr(df_both["sta_addr_master"])).to_numpy(bool)
same_lat  = same_coord(df_both["sta_lat"],  df_both["sta_lat_master"])
same_long = same_coord(df_both["sta_long"], df_both["sta_long_master"])
all_same  = same_addr & same_lat & same_long

n_total, n_diff = len(df_both), int((~all_same).sum())
print(f"비교 대상 {n_total:,}행 / 동일 {n_total - n_diff:,}행 / 불일치 {n_diff:,}행")

# 3) 불일치가 있으면 먼저 df_mismatch에 담기 (없으면 만들지 않음)
if n_diff > 0:
    df_mismatch = df_both.loc[~all_same].copy()
    df_mismatch["diff_addr"] = ~same_addr[~all_same]
    df_mismatch["diff_lat"]  = ~same_lat[~all_same]
    df_mismatch["diff_long"] = ~same_long[~all_same]

    names = np.array(["주소", "위도", "경도"])
    mask = df_mismatch[["diff_addr", "diff_lat", "diff_long"]].to_numpy(dtype=bool)
    df_mismatch["diff_cols"] = pd.Series([",".join(names[r]) for r in mask],
                                         index=df_mismatch.index, dtype="string")   # 예: "주소,위도"
    df_mismatch["lat_gap"]  = (df_mismatch["sta_lat"]  - df_mismatch["sta_lat_master"]).abs()
    df_mismatch["long_gap"] = (df_mismatch["sta_long"] - df_mismatch["sta_long_master"]).abs()
    df_mismatch = df_mismatch.reset_index(drop=True)

    print("\n불일치 항목별 건수:\n", df_mismatch["diff_cols"].value_counts().to_string())
    display(df_mismatch.head(10))
else:
    print("→ 모두 동일: df_mismatch는 만들지 않습니다.")

# 4) df_both를 7개 컬럼만 남기기 (불일치 행은 옵션에 따라 제외)
if n_diff > 0 and REMOVE_MISMATCH_FROM_BOTH:
    df_both = df_both.loc[all_same, keep_cols].reset_index(drop=True)
else:
    df_both = df_both[keep_cols].reset_index(drop=True)

print(f"\ndf_both: {len(df_both):,}행 × {df_both.shape[1]}컬럼")
print(df_both.dtypes)
display(df_both.head())

비교 대상 3,244행 / 동일 3,244행 / 불일치 0행
→ 모두 동일: df_mismatch는 만들지 않습니다.

df_both: 3,244행 × 7컬럼
sta_loc     string[python]
rent_id     string[python]
rent_no              Int64
rent_nm     string[python]
sta_addr    string[python]
sta_lat            float64
sta_long           float64
dtype: object


,sta_loc,rent_id,rent_no,rent_nm,sta_addr,sta_lat,sta_long
0,마포구,ST-10,108,서교동 사거리,"서울특별시 마포구 양화로 93 , 427",37.552746,126.918617
1,광진구,ST-100,503,더샵스타시티 C동 앞,"서울특별시 광진구 아차산로 262 , 더샵스타시티 C동 앞",37.536667,127.073593
2,양천구,ST-1000,729,서부식자재마트 건너편,"서울특별시 양천구 신정동 236 , 서부식자재마트 건너편",37.510380,126.866798
3,양천구,ST-1002,731,서울시 도로환경관리센터,"서울특별시 양천구 목동동로 316-6 , 서울시 도로환경관리센터",37.529900,126.876541
4,양천구,ST-1003,732,신월중학교,"서울특별시 양천구 화곡로 59 , 신월동 이마트",37.539551,126.828300


### 교통정보시스템에만 있는 정류소는 나중에 확인하기.

In [ ]:
df_diff['data_source'].unique()

<StringArray>
['교통정보시스템']
Length: 1, dtype: string

####db저장_ 정류장 (양쪽일치 station_both / 한쪽만 station_diff)

In [ ]:
# ============================================================
# [정류장 저장] 전처리된 정류장 DataFrame을 타입 그대로 DB에 저장
#   station_both : 따릉이·교통정보시스템 양쪽에 있고 값이 동일한 정류장 (df_both)
#   station_diff : 한쪽에만 있는 정류장, data_source 컬럼으로 출처 구분 (df_diff)
# ============================================================
ensure_conn()

save_df_typed(df_both, "station_both", conn, if_exists="replace", pk="rent_id")
save_df_typed(df_diff, "station_diff", conn, if_exists="replace", pk="rent_id")

# 저장된 컬럼 타입 확인
for t in ["station_both", "station_diff"]:
    print(f"\n[{t}]")
    display(query_df("""SELECT column_name, data_type FROM information_schema.columns
                        WHERE table_schema = current_schema() AND table_name = %s
                        ORDER BY ordinal_position""", [t]))

#[참고_station_both]
# "sta_loc",   # 행정구
# "rent_id",   # 정류장 아이디 (기본키)
# "rent_no",   # 정류장 번호
# "rent_nm",   # 정류장 이름
# "sta_addr",  # 주소 (주소1 , 주소2)
# "sta_lat",   # 위도
# "sta_long",  # 경도


STATION_BOTH: 3,244행 저장 완료
STATION_DIFF: 188행 저장 완료

[station_both]


,column_name,data_type
0,sta_loc,text
1,rent_id,text
2,rent_no,integer
3,rent_nm,text
4,sta_addr,text
5,sta_lat,double precision
6,sta_long,double precision



[station_diff]


,column_name,data_type
0,sta_loc,text
1,rent_id,text
2,rent_no,integer
3,rent_nm,text
4,sta_addr,text
5,sta_lat,double precision
6,sta_long,double precision
7,data_source,text


##1-2. 지하철역 - 따릉이 정류장 근접

* data_processing 노트북 맨 아래에 이어 붙여서 실행 (앞의 DB 연결, query_df, save_df_typed, exec_sql, df_mismatch 사용)
* 지도 API는 사용하지 않음 : 지하철역과 정류장 모두 DB에 위도·경도가 있으므로 좌표로 직접 계산
* 근처 기준 : 직선거리(haversine) 300m 이내 / 300~500m, 500m 초과는 연결하지 않음
* 사용하는 테이블 : raw_subway_station_master, station_both, station_diff
* 만들 테이블 : station_subway_link (반경 안의 정류장-지하철역 쌍)

In [ ]:
# 설정값과 공통 함수
NEAR_RADIUS_M     = 300     # 가까움 구간 경계
MAX_LINK_RADIUS_M = 500     # 연결 최대 반경
SAME_STATION_M    = 700     # 같은 이름의 역이 이 거리 이상 벌어지면 다른 역으로 분리

MANUAL_NAME_MAP = {}        # 역명 표기가 다를 때 수동 매핑 예) {"총신대입구(이수)": "이수"}


def haversine_m(lat1, lon1, lat2, lon2):
    """두 좌표 사이 직선거리(m). numpy 배열 브로드캐스팅 지원"""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371008.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def normalize_name(name):
    """역명 정규화: 수동 매핑 → 괄호 내용 제거 → 공백 제거 → 끝의 '역' 제거"""
    s = str(name).strip()
    s = MANUAL_NAME_MAP.get(s, s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s+", "", s)
    return re.sub(r"역$", "", s)


def normalize_line(line):
    """호선 정규화: '01호선', '1호선', '1' → '1호선' / 그 외 노선명은 공백만 제거"""
    s = str(line).strip()
    m = re.fullmatch(r"0*(\d+)\s*(호선)?", s)
    return f"{int(m.group(1))}호선" if m else re.sub(r"\s+", "", s)


def swap_fix(df, lat_col, lon_col):
    """위도 자리에 126~127, 경도 자리에 37 이 들어간 행(뒤바뀜)을 교정. 교정한 행 수 반환"""
    swapped = df[lat_col].between(124, 132) & df[lon_col].between(33, 39)
    df.loc[swapped, [lat_col, lon_col]] = df.loc[swapped, [lon_col, lat_col]].to_numpy()
    return int(swapped.sum())

###1-2-1. 지하철역

In [ ]:
#샘플 확인
df_sample = query_df("SELECT * FROM raw_subway_station_master LIMIT 10")
display(df_sample)
print(df_sample.dtypes)

[재접속 완료]


,bldn_id,bldn_nm,route,lat,lot
0,0150,서울역,1호선,37.556228,126.972135
1,0151,시청,1호선,37.565715,126.977088
2,0152,종각,1호선,37.570161,126.982923
3,0153,종로3가,1호선,37.570406,126.991847
4,0154,종로5가,1호선,37.570926,127.001849
5,0155,동대문,1호선,37.571687,127.01093
6,0156,신설동,1호선,37.576048,127.024634
7,0157,제기동,1호선,37.578103,127.034893
8,0158,청량리(서울시립대입구),1호선,37.579956,127.044585
9,0159,동묘앞,1호선,37.573197,127.01648


bldn_id    object
bldn_nm    object
route      object
lat        object
lot        object
dtype: object


In [ ]:
#필요한 컬럼 조회
df_subway = query_df("""
    SELECT bldn_id,      -- 역 아이디
           bldn_nm,      -- 역명
           route,        -- 호선
           lat,          -- 위도
           lot           -- 경도
    FROM raw_subway_station_master
""")

print(df_subway.shape)
print(df_subway.dtypes)         # 전부 object(문자열)로 나옴
display(df_subway.head())

(784, 5)
bldn_id    object
bldn_nm    object
route      object
lat        object
lot        object
dtype: object


,bldn_id,bldn_nm,route,lat,lot
0,0150,서울역,1호선,37.556228,126.972135
1,0151,시청,1호선,37.565715,126.977088
2,0152,종각,1호선,37.570161,126.982923
3,0153,종로3가,1호선,37.570406,126.991847
4,0154,종로5가,1호선,37.570926,127.001849


In [ ]:
#지하철역 전처리
df_subway2 = df_subway.copy()

# 1) 문자열 컬럼: 문자열 타입으로 바꾸고 앞뒤 공백 제거
for c in ["bldn_id", "bldn_nm", "route"]:
    df_subway2[c] = df_subway2[c].astype("string").str.strip()

# 2) 위도/경도: 실수로 변환. 변환 불가 값은 NaN
df_subway2["lat"] = pd.to_numeric(df_subway2["lat"], errors="coerce").astype("float64")
df_subway2["lot"] = pd.to_numeric(df_subway2["lot"], errors="coerce").astype("float64")

# ---------------- 변환 결과 점검 ----------------
print(df_subway2.dtypes, "\n")
print("결측 수:\n", df_subway2.isna().sum(), "\n")
print("좌표가 0이거나 결측:", int(((df_subway2["lat"] == 0) | (df_subway2["lot"] == 0)
                               | df_subway2["lat"].isna() | df_subway2["lot"].isna()).sum()), "건")
print("bldn_id 중복:", int(df_subway2["bldn_id"].duplicated().sum()), "건")
display(df_subway2.head(10))

bldn_id    string[python]
bldn_nm    string[python]
route      string[python]
lat               float64
lot               float64
dtype: object 

결측 수:
 bldn_id    0
bldn_nm    0
route      0
lat        0
lot        0
dtype: int64 

좌표가 0이거나 결측: 0 건
bldn_id 중복: 0 건


,bldn_id,bldn_nm,route,lat,lot
0,0150,서울역,1호선,37.556228,126.972135
1,0151,시청,1호선,37.565715,126.977088
2,0152,종각,1호선,37.570161,126.982923
3,0153,종로3가,1호선,37.570406,126.991847
4,0154,종로5가,1호선,37.570926,127.001849
5,0155,동대문,1호선,37.571687,127.010930
6,0156,신설동,1호선,37.576048,127.024634
7,0157,제기동,1호선,37.578103,127.034893
8,0158,청량리(서울시립대입구),1호선,37.579956,127.044585
9,0159,동묘앞,1호선,37.573197,127.016480


In [ ]:
# 좌표검증과 물리역
df_subway3 = df_subway2.copy()
n_swapped = swap_fix(df_subway3, "lat", "lot")

valid = df_subway3["lat"].between(37.0, 38.2) & df_subway3["lot"].between(126.0, 128.0)
df_subway_err = df_subway3[~valid].copy()
df_subway_pt = df_subway3[valid].drop_duplicates("bldn_id").reset_index(drop=True)

df_subway_pt["name_key"] = df_subway_pt["bldn_nm"].map(normalize_name)
df_subway_pt["line_key"] = df_subway_pt["route"].map(normalize_line)

df_subway_pt["phys_key"] = df_subway_pt["name_key"]
for name_key, g in df_subway_pt.groupby("name_key"):
    spread = haversine_m(g["lat"], g["lot"], g["lat"].mean(), g["lot"].mean()).max()
    if spread > SAME_STATION_M:
        df_subway_pt.loc[g.index, "phys_key"] = name_key + "_" + g["line_key"]

df_subway_station = (
    df_subway_pt.groupby("phys_key")
    .agg(subway_name=("bldn_nm", "first"),
         lat=("lat", "mean"), lot=("lot", "mean"),
         lines=("line_key", lambda s: ", ".join(sorted(s.unique()))),
         n_lines=("line_key", "nunique"))
    .reset_index()
)

print(f"원본 {len(df_subway):,}행 → 유효 {len(df_subway_pt):,}행 / 좌표 오류 {len(df_subway_err)}행 / 위경도 뒤바뀜 교정 {n_swapped}행")
print(f"물리역 {len(df_subway_station):,}곳 / 환승역(2개 호선 이상) {(df_subway_station['n_lines'] >= 2).sum():,}곳")
if len(df_subway_err):
    print("\n좌표 오류 역:"); display(df_subway_err.head(20))
display(df_subway_station.head())

원본 784행 → 유효 772행 / 좌표 오류 12행 / 위경도 뒤바뀜 교정 0행
물리역 643곳 / 환승역(2개 호선 이상) 109곳

좌표 오류 역:


,bldn_id,bldn_nm,route,lat,lot
219,1401,봉명,장항선,36.801215,127.135763
220,1402,쌍용(나사렛대),장항선,36.793759,127.121400
221,1403,아산,장항선,36.792053,127.104361
222,1404,탕정,장항선,36.788660,127.084850
223,1405,배방,장항선,36.777629,127.052991
224,1407,온양온천,장항선,36.780483,127.003249
225,1408,신창(순천향대),장항선,36.769502,126.951108
270,1724,평택,경부선,36.990726,127.085159
271,1725,성환,경부선,36.916076,127.126964
272,1726,직산,경부선,36.870593,127.143904


,phys_key,subway_name,lat,lot,lines,n_lines
0,4.19민주묘지,4.19민주묘지,37.649502,127.013684,우이신설선,1
1,가능,가능,37.748577,127.044213,경원선,1
2,가락시장,가락시장,37.492566,127.118077,"3호선, 8호선",2
3,가산디지털단지,가산디지털단지,37.480959,126.882619,"7호선, 경부선",2
4,가양,가양,37.561391,126.854456,9호선,1


###1-2-2. 따릉이 정류장.


* station_both : 양쪽에 있고 값이 동일한 정류장 → 전부 사용
* station_diff 중 data_source = '따릉이' : 따릉이에만 있는 정류장 → 사용 (구분 컬럼 station_source)
* station_diff 중 data_source = '교통정보시스템' : 나중에 확인할 정류장 → 제외
* 불일치로 df_mismatch에 분리된 정류장은 두 테이블에 없어 제외됨

In [ ]:
# 정류장 불러오기
df_st_both = query_df("SELECT rent_id, rent_no, rent_nm, sta_lat, sta_long FROM station_both")
df_st_both["station_source"] = "양쪽일치"

df_st_diff = query_df("""SELECT rent_id, rent_no, rent_nm, sta_lat, sta_long
                         FROM station_diff WHERE data_source = '따릉이'""")
df_st_diff["station_source"] = "따릉이만"

df_bike_st = pd.concat([df_st_both, df_st_diff], ignore_index=True)
df_bike_st["rent_no"]   = pd.to_numeric(df_bike_st["rent_no"], errors="coerce").astype("Int64")
df_bike_st["sta_lat"]   = pd.to_numeric(df_bike_st["sta_lat"], errors="coerce").astype("float64")
df_bike_st["sta_long"]  = pd.to_numeric(df_bike_st["sta_long"], errors="coerce").astype("float64")

# 위경도 뒤바뀜 교정 후, 좌표가 0 / 결측 / 범위 밖인 정류장은 제외
n_swapped = swap_fix(df_bike_st, "sta_lat", "sta_long")
valid = df_bike_st["sta_lat"].between(37.0, 38.2) & df_bike_st["sta_long"].between(126.0, 128.0)
df_bike_st_err = df_bike_st[~valid].copy()
df_bike_st = df_bike_st[valid].reset_index(drop=True)

# ---------------- 점검 ----------------
print(f"station_both {len(df_st_both):,}곳 + station_diff(따릉이만) {len(df_st_diff):,}곳")
print(f"→ 사용 {len(df_bike_st):,}곳 / 좌표 오류로 제외 {len(df_bike_st_err)}곳 / 위경도 뒤바뀜 교정 {n_swapped}곳")
print("rent_id 중복:", int(df_bike_st["rent_id"].duplicated().sum()), "건")
if "df_mismatch" in globals():
    print(f"※ 불일치 정류장(df_mismatch) {len(df_mismatch):,}곳은 포함되지 않았습니다.")
if len(df_bike_st_err):
    print("\n좌표 오류 정류장:"); display(df_bike_st_err.head(20))
display(df_bike_st["station_source"].value_counts().to_frame())

station_both 3,244곳 + station_diff(따릉이만) 0곳
→ 사용 3,241곳 / 좌표 오류로 제외 3곳 / 위경도 뒤바뀜 교정 0곳
rent_id 중복: 0 건

좌표 오류 정류장:


/tmp/ipykernel_1483/2471462793.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_bike_st = pd.concat([df_st_both, df_st_diff], ignore_index=True)


,rent_id,rent_no,rent_nm,sta_lat,sta_long,station_source
90,ST-1090,1272,송파 글마루도서관,0.0,0.0,양쪽일치
2838,ST-601,1603,롯데캐슬 102동 코너(월계주유소건너),0.0,0.0,양쪽일치
3088,ST-844,1309,보문3교 옆,0.0,0.0,양쪽일치


,count
station_source,
양쪽일치,3241


####db저장_ 좌표오류로 조회불가 정류장

In [ ]:
# ============================================================
# [좌표 오류 정류장 → df_diff] df_bike_st_err 를 df_diff 와 station_diff 테이블에 추가
# ============================================================
BOTH_SOURCE      = "따릉이,교통정보시스템"   # 두 곳 모두에 있는 정류장의 data_source 값
REMOVE_FROM_BOTH = False                    # True: 옮긴 정류장을 station_both 에서도 삭제 / False: 복사만

diff_cols = [
    "sta_loc",      # 행정구
    "rent_id",      # 정류장 아이디 (기본키)
    "rent_no",      # 정류장 번호
    "rent_nm",      # 정류장 이름
    "sta_addr",     # 주소 (주소1 , 주소2)
    "sta_lat",      # 위도 (좌표 오류 정류장은 0 또는 결측)
    "sta_long",     # 경도 (좌표 오류 정류장은 0 또는 결측)
    "data_source",  # 데이터 출처 (따릉이 / 교통정보시스템 / 따릉이,교통정보시스템 = 두 곳에 있으나 좌표오류로 조회불가)
]

if "df_bike_st_err" not in globals():
    raise RuntimeError("df_bike_st_err 가 없습니다. 정류장 불러오기 셀을 먼저 실행하세요.")

# df_diff 는 정류장 비교 셀에서 만들어지는 변수 : 런타임을 다시 시작해서 없으면 DB 의 station_diff 에서 불러옴
if "df_diff" not in globals():
    df_diff = query_df(f"SELECT {', '.join(diff_cols)} FROM station_diff")
    for c in ["sta_loc", "rent_id", "rent_nm", "sta_addr", "data_source"]:
        df_diff[c] = df_diff[c].astype("string")
    df_diff["rent_no"]  = pd.to_numeric(df_diff["rent_no"], errors="coerce").astype("Int64")
    df_diff["sta_lat"]  = pd.to_numeric(df_diff["sta_lat"], errors="coerce").astype("float64")
    df_diff["sta_long"] = pd.to_numeric(df_diff["sta_long"], errors="coerce").astype("float64")
    print(f"※ df_diff 가 메모리에 없어 station_diff 에서 불러왔습니다: {len(df_diff):,}행")

err_both      = df_bike_st_err[df_bike_st_err["station_source"] == "양쪽일치"]
err_bike_only = df_bike_st_err[df_bike_st_err["station_source"] == "따릉이만"]
ids_both      = err_both["rent_id"].dropna().tolist()

# 1) station_both 에서 온 정류장은 행정구·주소까지 있는 원래 행을 다시 조회 (df_diff 와 같은 컬럼)
if ids_both:
    df_err_add = query_df("""SELECT sta_loc, rent_id, rent_no, rent_nm, sta_addr, sta_lat, sta_long
                            FROM station_both WHERE rent_id = ANY(%s)""", [ids_both])
    for c in ["sta_loc", "rent_id", "rent_nm", "sta_addr"]:
        df_err_add[c] = df_err_add[c].astype("string").str.strip()
    df_err_add["rent_no"]   = pd.to_numeric(df_err_add["rent_no"], errors="coerce").astype("Int64")
    df_err_add["sta_lat"]   = pd.to_numeric(df_err_add["sta_lat"], errors="coerce").astype("float64")
    df_err_add["sta_long"]  = pd.to_numeric(df_err_add["sta_long"], errors="coerce").astype("float64")
    df_err_add["data_source"] = pd.Series(BOTH_SOURCE, index=df_err_add.index, dtype="string")
else:
    df_err_add = pd.DataFrame(columns=diff_cols)

# 2) df_diff(메모리)에 추가 : 이미 들어 있는 rent_id 는 건너뜀
mem_new = df_err_add[~df_err_add["rent_id"].isin(df_diff["rent_id"])]
if len(mem_new):
    df_diff = pd.concat([df_diff, mem_new[diff_cols]], ignore_index=True)
    df_diff["data_source"] = df_diff["data_source"].astype("string")

# 3) station_diff(DB)에 추가 : 이미 저장된 rent_id 는 건너뜀
ensure_conn()
ids_in_db = set(query_df("SELECT rent_id FROM station_diff")["rent_id"])
db_new = df_err_add[~df_err_add["rent_id"].isin(ids_in_db)]
if len(db_new):
    # save_df_typed 의 append 를 쓰지 않고 직접 COPY (함수 정의 셀이 어떤 버전이어도 동작)
    buf = io.StringIO()
    db_new[diff_cols].to_csv(buf, index=False, header=False)
    buf.seek(0)
    try:
        with conn.cursor() as cur:
            cur.copy_expert(f"COPY station_diff ({', '.join(diff_cols)}) FROM STDIN WITH (FORMAT csv)", buf)
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    print(f"STATION_DIFF: {len(db_new):,}행 추가 저장 완료")

# 4) (선택) station_both 에서도 삭제
if REMOVE_FROM_BOTH and ids_both:
    exec_sql("DELETE FROM station_both WHERE rent_id = ANY(%s)", [ids_both], title="station_both 에서 삭제")
    if "df_both" in globals():                  # df_both 가 메모리에 있을 때만 같이 정리
        df_both = df_both[~df_both["rent_id"].isin(ids_both)].reset_index(drop=True)

# ---------------- 결과 점검 ----------------
print(f"좌표 오류 정류장 {len(df_bike_st_err)}곳 = 양쪽일치(station_both) {len(err_both)}곳 + 따릉이만(station_diff) {len(err_bike_only)}곳")
print(f"  양쪽일치 → df_diff 에 새로 추가 {len(mem_new)}곳 / station_diff 에 새로 저장 {len(db_new)}곳 (data_source = {BOTH_SOURCE})")
n_in = int(err_bike_only["rent_id"].isin(df_diff["rent_id"]).sum())
print(f"  따릉이만 → 이미 df_diff 에 있음 {n_in}곳 / 없음 {len(err_bike_only) - n_in}곳")
if len(err_bike_only) - n_in:
    print("  ⚠ df_diff 에 없는 '따릉이만' 정류장이 있습니다. df_diff / station_diff 를 다시 만들었는지 확인하세요.")
print(f"  station_both 에서 삭제: {'함' if REMOVE_FROM_BOTH else '안 함 (REMOVE_FROM_BOTH=False)'}")
display(mem_new if len(mem_new) else df_diff[df_diff["data_source"] == BOTH_SOURCE])
display(query_df("SELECT data_source, COUNT(*) AS cnt FROM station_diff GROUP BY 1 ORDER BY 1"))

STATION_DIFF: 3행 추가 저장 완료
좌표 오류 정류장 3곳 = 양쪽일치(station_both) 3곳 + 따릉이만(station_diff) 0곳
  양쪽일치 → df_diff 에 새로 추가 0곳 / station_diff 에 새로 저장 3곳 (data_source = 따릉이,교통정보시스템)
  따릉이만 → 이미 df_diff 에 있음 0곳 / 없음 0곳
  station_both 에서 삭제: 안 함 (REMOVE_FROM_BOTH=False)


,sta_loc,rent_id,rent_no,rent_nm,sta_addr,sta_lat,sta_long,data_source
188,송파구,ST-1090,1272,송파 글마루도서관,"서울특별시 송파구 충민로 120 , 송파 글마루도서관",0.0,0.0,"따릉이,교통정보시스템"
189,노원구,ST-601,1603,롯데캐슬 102동 코너(월계주유소건너),"서울특별시 노원구 월계로 252 , 노원구 월계동 860-16",0.0,0.0,"따릉이,교통정보시스템"
190,성북구,ST-844,1309,보문3교 옆,"서울특별시 성북구 인촌로 16 , 성북구 안암동2가 140-1",0.0,0.0,"따릉이,교통정보시스템"


,data_source,cnt
0,교통정보시스템,188
1,"따릉이,교통정보시스템",3


#### 역- 지하철 근접계산

In [ ]:
# 근접 계산
dist = haversine_m(
    df_bike_st["sta_lat"].to_numpy()[:, None],
    df_bike_st["sta_long"].to_numpy()[:, None],
    df_subway_station["lat"].to_numpy()[None, :],
    df_subway_station["lot"].to_numpy()[None, :],
)                                                   # shape = (정류장 수, 물리역 수)

bi, si = np.where(dist <= MAX_LINK_RADIUS_M)
df_link = pd.DataFrame({
    "rent_id":        df_bike_st["rent_id"].to_numpy()[bi],
    "rent_no":        df_bike_st["rent_no"].to_numpy()[bi],
    "rent_nm":        df_bike_st["rent_nm"].to_numpy()[bi],
    "station_source": df_bike_st["station_source"].to_numpy()[bi],
    "phys_key":       df_subway_station["phys_key"].to_numpy()[si],
    "subway_name":    df_subway_station["subway_name"].to_numpy()[si],
    "lines":          df_subway_station["lines"].to_numpy()[si],
    "distance_m":     dist[bi, si].round(1),
})
df_link["radius_band"] = np.where(df_link["distance_m"] <= NEAR_RADIUS_M,
                                  f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m")
# 한 정류장이 여러 역 반경에 걸칠 때 가까운 순서
df_link["subway_rank"] = df_link.groupby("rent_id")["distance_m"].rank(method="first").astype(int)
df_link["is_nearest"]  = df_link["subway_rank"].eq(1)
df_link["radius_m"]    = MAX_LINK_RADIUS_M
df_link = df_link.sort_values(["rent_id", "subway_rank"]).reset_index(drop=True)

# 반경 밖 정류장도 포함한 '가장 가까운 역까지 거리' (확인용)
nearest_m = pd.Series(dist.min(axis=1), index=df_bike_st["rent_id"])

print(f"연결 쌍 {len(df_link):,}건 / 역 근처 정류장 {df_link['rent_id'].nunique():,}곳 "
      f"/ 근처 정류장이 있는 역 {df_link['phys_key'].nunique():,}곳")
display(df_link.head(10))

연결 쌍 2,178건 / 역 근처 정류장 1,919곳 / 근처 정류장이 있는 역 309곳


,rent_id,rent_no,rent_nm,station_source,phys_key,subway_name,lines,distance_m,radius_band,subway_rank,is_nearest,radius_m
0,ST-1000,729,서부식자재마트 건너편,양쪽일치,양천구청,양천구청,2호선,240.5,0-300m,1,True,500
1,ST-1008,737,장수공원,양쪽일치,신정네거리,신정네거리,2호선,422.2,300-500m,1,True,500
2,ST-1013,742,등촌역 5번 출구 뒤,양쪽일치,등촌,등촌,9호선,98.5,0-300m,1,True,500
3,ST-1015,744,신목동역 2번 출구,양쪽일치,신목동,신목동,9호선,67.5,0-300m,1,True,500
4,ST-102,505,자양사거리 광진아크로텔 앞,양쪽일치,구의,구의(광진구청),2호선,323.8,300-500m,1,True,500
5,ST-1021,935,역촌역 1번 출구,양쪽일치,역촌,역촌,6호선,106.8,0-300m,1,True,500
6,ST-1032,946,독바위역,양쪽일치,독바위,독바위,6호선,13.1,0-300m,1,True,500
7,ST-1033,947,연신내 선일하이츠빌 정류소,양쪽일치,연신내,연신내,"3호선, 6호선, 수도권광역급행철도",430.2,300-500m,1,True,500
8,ST-1034,948,디지털미디어 시티역 4번출구(DMC역),양쪽일치,디지털미디어시티,디지털미디어시티,"6호선, 경의중앙선, 공항철도1호선",197.9,0-300m,1,True,500
9,ST-1035,949,연신내역 1번 출구,양쪽일치,연신내,연신내,"3호선, 6호선, 수도권광역급행철도",101.3,0-300m,1,True,500


In [ ]:
# 기준 확인
for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 정류장: {(nearest_m <= r).mean():.1%}")
print(f"반경 {MAX_LINK_RADIUS_M}m 안에 역이 없는 정류장: {(nearest_m > MAX_LINK_RADIUS_M).sum():,}곳 / {len(df_bike_st):,}곳")

print("\n가장 가까운 역까지 거리(m) 분포")
display(nearest_m.describe().round(0).to_frame("nearest_m"))
print("역당 근처 정류장 수 분포")
display(df_link.groupby("phys_key").size().describe().round(1).to_frame("정류장 수"))

# 좌표가 한쪽만 틀어진 경우 모든 정류장이 '반경 밖'으로 나옴 → 중심 간 거리로 점검
gap_km = haversine_m(df_bike_st["sta_lat"].mean(), df_bike_st["sta_long"].mean(),
                     df_subway_station["lat"].mean(), df_subway_station["lot"].mean()) / 1000
print(f"\n정류장 중심 ↔ 역 중심 거리: {gap_km:.1f} km  (서울 안이면 수 km 이내가 정상)")
if (nearest_m > MAX_LINK_RADIUS_M).mean() > 0.9 or gap_km > 10:
    print("⚠ 거의 모든 정류장이 반경 밖입니다. 컬럼명(lat/lot)과 좌표 전처리를 다시 확인하세요.")

check_name = "합정"          # 상식 점검 : 아는 역을 넣어 보기
print(f"\n'{check_name}' 근처 정류장 (가까운 순)")
display(df_link[df_link["subway_name"].str.contains(check_name, na=False)].sort_values("distance_m").head(10))

가장 가까운 역이 300m 이내인 정류장: 34.1%
가장 가까운 역이 500m 이내인 정류장: 59.2%
반경 500m 안에 역이 없는 정류장: 1,322곳 / 3,241곳

가장 가까운 역까지 거리(m) 분포


,nearest_m
count,3241.0
mean,496.0
std,383.0
min,3.0
25%,224.0
50%,427.0
75%,669.0
max,3139.0


역당 근처 정류장 수 분포


,정류장 수
count,309.0
mean,7.0
std,3.5
min,1.0
25%,5.0
50%,7.0
75%,9.0
max,23.0



정류장 중심 ↔ 역 중심 거리: 3.5 km  (서울 안이면 수 km 이내가 정상)

'합정' 근처 정류장 (가까운 순)


,rent_id,rent_no,rent_nm,station_source,phys_key,subway_name,lines,distance_m,radius_band,subway_rank,is_nearest,radius_m
2048,ST-8,106,합정역 7번출구 앞,양쪽일치,합정,합정,"2호선, 6호선",101.7,0-300m,1,True,500
1979,ST-7,105,합정역 5번출구 앞,양쪽일치,합정,합정,"2호선, 6호선",132.4,0-300m,1,True,500
1911,ST-6,104,합정역 1번출구 앞,양쪽일치,합정,합정,"2호선, 6호선",204.7,0-300m,1,True,500
777,ST-2156,497,합정동주민센터 앞,양쪽일치,합정,합정,"2호선, 6호선",292.1,0-300m,1,True,500
1087,ST-2534,4211,시석빌딩 앞,양쪽일치,합정,합정,"2호선, 6호선",296.2,0-300m,1,True,500


####db저장_ 정류장-지하철역 근접 관계 (station_subway_link)

In [ ]:
link_cols = [
    "rent_id",         # 정류장 아이디
    "rent_no",         # 정류장 번호
    "rent_nm",         # 정류장 이름
    "station_source",  # 정류장 출처 (양쪽일치 / 따릉이만)
    "phys_key",        # 물리역 키 (환승역은 1개로 묶음)
    "subway_name",     # 지하철역 이름
    "lines",           # 지나가는 호선
    "distance_m",      # 정류장-역 직선거리(m)
    "radius_band",     # 거리 구간 (0-300m / 300-500m)
    "subway_rank",     # 그 정류장에서 가까운 역 순서 (1 = 가장 가까움)
    "is_nearest",      # 가장 가까운 역 여부 (True / False)
    "radius_m",        # 연결에 사용한 최대 반경(m)
]
ensure_conn()
save_df_typed(df_link[link_cols], "station_subway_link", conn, if_exists="replace")
exec_sql("ALTER TABLE station_subway_link ADD PRIMARY KEY (rent_id, phys_key)", title="기본키 지정")
exec_sql("CREATE INDEX IF NOT EXISTS idx_station_subway_link_phys ON station_subway_link (phys_key)", title="인덱스 생성")

# 저장된 컬럼 타입 확인
display(query_df("""SELECT column_name, data_type FROM information_schema.columns
                    WHERE table_schema = current_schema() AND table_name = 'station_subway_link'
                    ORDER BY ordinal_position"""))

STATION_SUBWAY_LINK: 2,178행 저장 완료
✔ 기본키 지정 완료 (1초)
✔ 인덱스 생성 완료 (1초)


,column_name,data_type
0,rent_id,text
1,rent_no,integer
2,rent_nm,text
3,station_source,text
4,phys_key,text
5,subway_name,text
6,lines,text
7,distance_m,double precision
8,radius_band,text
9,subway_rank,integer


In [ ]:
# 저장결과 확인
chk = query_df("""SELECT COUNT(*) AS n, COUNT(DISTINCT rent_id) AS rents, COUNT(DISTINCT phys_key) AS subways,
                         SUM(is_nearest::int) AS nearest FROM station_subway_link""")
print(chk.to_string(index=False))
print("DataFrame 과 행 수:", "일치 ✔" if int(chk["n"][0]) == len(df_link) else "불일치 ⚠")

display(query_df("""SELECT radius_band, COUNT(*) AS cnt, ROUND(AVG(distance_m)::numeric, 1) AS avg_m
                    FROM station_subway_link GROUP BY 1 ORDER BY 1"""))
display(query_df("SELECT * FROM station_subway_link ORDER BY rent_id, subway_rank LIMIT 5"))

   n  rents  subways  nearest
2178   1919      309     1919
DataFrame 과 행 수: 일치 ✔


,radius_band,cnt,avg_m
0,0-300m,1119,153.6
1,300-500m,1059,409.4


,rent_id,rent_no,rent_nm,station_source,phys_key,subway_name,lines,distance_m,radius_band,subway_rank,is_nearest,radius_m
0,ST-1000,729,서부식자재마트 건너편,양쪽일치,양천구청,양천구청,2호선,240.5,0-300m,1,True,500
1,ST-1008,737,장수공원,양쪽일치,신정네거리,신정네거리,2호선,422.2,300-500m,1,True,500
2,ST-1013,742,등촌역 5번 출구 뒤,양쪽일치,등촌,등촌,9호선,98.5,0-300m,1,True,500
3,ST-1015,744,신목동역 2번 출구,양쪽일치,신목동,신목동,9호선,67.5,0-300m,1,True,500
4,ST-102,505,자양사거리 광진아크로텔 앞,양쪽일치,구의,구의(광진구청),2호선,323.8,300-500m,1,True,500


# 여기!

##2\. 대여이력 관련 전처리

In [6]:
# 값의 형식(날짜 문자열 모양, 성별 값 등)을 확인해야 전처리 방향을 정할 수 있음
df_sample = query_df("SELECT * FROM rental_history_raw LIMIT 10")
display(df_sample)
print(df_sample.dtypes)        # 컬럼이 전부 TEXT라 object(문자열)로 나옴

,bike_no,rental_date,rental_station_no,rental_station_nm,rental_hold,return_date,return_station_no,return_station_nm,return_hold,rental_time,rental_distance,birth,sex,user_type,rental_station_id,return_station_id,bicycle_type
0,SPB-38113,2025-07-26 04:40:04,04540,목동역 3번 출구,0,2025-07-26 06:24:59,04547,목동역 2번 출구,0,104,14873.47,1985,F,내국인,ST-3104,ST-3304,일반자전거
1,SPB-71753,2025-07-26 06:07:31,04397,마장축산물시장 북문 건너편,0,2025-07-26 06:25:02,00344,성균관대 E하우스 앞,0,17,3475.82,1963,M,내국인,ST-3321,ST-160,일반자전거
2,SPB-69574,2025-07-26 06:17:49,00602,장안동 사거리,0,2025-07-26 06:25:06,04108,답십리파크자이아파트 앞,0,7,1524.09,1987,M,내국인,ST-262,ST-2501,일반자전거
3,SPB-56758,2025-07-26 06:05:44,03772,길성그랑프리텔아파트,0,2025-07-26 06:25:10,05095,원당근린공원,0,19,2883.49,1993,F,내국인,ST-2482,ST-3226,일반자전거
4,SPB-69309,2025-07-26 06:14:56,03131,연가교 교차로,0,2025-07-26 06:25:10,00103,망원역 2번출구 앞,0,10,2387.46,1978,F,내국인,ST-2216,ST-5,일반자전거
5,SPB-39805,2025-07-26 05:40:56,01251,석촌역 2번출구,0,2025-07-26 06:25:11,01251,석촌역 2번출구,0,44,7559.08,1989,F,내국인,ST-1070,ST-1070,일반자전거
6,SPB-54399,2025-07-26 06:05:11,02297,남부터미널역 1번출구,0,2025-07-26 06:25:15,02104,사당역 5번출구,0,20,3522.24,1976,F,내국인,ST-1404,ST-712,일반자전거
7,SPB-69750,2025-07-26 06:05:20,01118,증미역 3번출구뒤(등촌두산위브센티움오피스텔),0,2025-07-26 06:25:20,02725,서울식물원 식물문화센터,0,20,2764.60,1968,M,내국인,ST-516,ST-2042,일반자전거
8,SPB-52433,2025-07-26 04:44:28,04007,노원 을지대학교 병원,0,2025-07-26 06:25:21,04080,창동주공19단지앞,0,100,5366.83,1996,\N,내국인,ST-2702,ST-2967,일반자전거
9,SPB-60102,2025-07-26 06:16:28,01910,낙원교회 맞은편,0,2025-07-26 06:25:21,01960,구로거리공원 동측(신도림역 방면),0,8,833.78,1993,M,내국인,ST-667,ST-1222,일반자전거


bike_no              object
rental_date          object
rental_station_no    object
rental_station_nm    object
rental_hold          object
return_date          object
return_station_no    object
return_station_nm    object
return_hold          object
rental_time          object
rental_distance      object
birth                object
sex                  object
user_type            object
rental_station_id    object
return_station_id    object
bicycle_type         object
dtype: object


###2-1. 용량 큰 데이터의 작업가능성 확인 _멈춰서 분리해야함.

In [7]:
# ============================================================
# [대여이력 점검] DB 안에서 집계 → 작은 결과만 코랩으로 가져옴
#   SRC: 처음엔 1% 표본(빠름), 확인 후 전체 테이블로 변경
# ============================================================
SRC = "rental_history_raw TABLESAMPLE SYSTEM (1)"     # 전체: "rental_history_raw"

# A) 자전거별 유형 (결과 행 수 ≈ 자전거 수라 작음)
bike_stat = query_df(f"""
    SELECT bike_no, bicycle_type, COUNT(*) AS cnt,
           MIN(rental_date) AS first_dt, MAX(rental_date) AS last_dt
    FROM {SRC} GROUP BY bike_no, bicycle_type
""")
n_type = bike_stat.groupby("bike_no")["bicycle_type"].nunique()
print("자전거 수:", bike_stat["bike_no"].nunique())
print("유형이 2개 이상인 자전거:", int((n_type > 1).sum()))
print(bike_stat.groupby("bicycle_type")["cnt"].sum().sort_values(ascending=False), "\n")

# B) 컬럼별 품질 한 번에 점검 (표본/전체 한 번 훑는 집계 쿼리)
profile = query_df(r"""
    SELECT COUNT(*) AS total,
           MIN(rental_date) AS min_rental, MAX(rental_date) AS max_rental,
           MIN(return_date) AS min_return, MAX(return_date) AS max_return,
           COUNT(*) FILTER (WHERE sex = '\N')                          AS sex_nullmark,
           COUNT(*) FILTER (WHERE birth = '\N')                        AS birth_nullmark,
           COUNT(*) FILTER (WHERE birth !~ '^\d{4}$' AND birth <> '\N') AS birth_bad,
           COUNT(*) FILTER (WHERE rental_date !~ '^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$') AS rental_date_bad,
           COUNT(*) FILTER (WHERE return_date !~ '^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$') AS return_date_bad,
           COUNT(*) FILTER (WHERE rental_time !~ '^\d+$')              AS time_bad,
           COUNT(*) FILTER (WHERE rental_distance !~ '^-?\d+(\.\d+)?$') AS dist_bad,
           COUNT(*) FILTER (WHERE rental_hold !~ '^\d+$')              AS rhold_bad,
           COUNT(*) FILTER (WHERE return_hold !~ '^\d+$')              AS return_hold_bad,
           COUNT(*) - COUNT(bike_no)           AS bike_no_null,
           COUNT(*) - COUNT(bicycle_type)      AS type_null,
           COUNT(*) - COUNT(rental_station_id) AS rstation_null,
           COUNT(*) - COUNT(return_station_id) AS return_station_null,
           COUNT(*) - COUNT(return_date)       AS return_date_null
    FROM """ + SRC)
print(profile.T.to_string(), "\n")

# C) 범주형 값 분포 (성별, 이용자 종류)
cat = query_df(f"SELECT sex, user_type, COUNT(*) AS cnt FROM {SRC} GROUP BY sex, user_type ORDER BY cnt DESC")
display(cat)

KeyboardInterrupt: 

In [8]:
# ============================================================
# [DB 실행 헬퍼] DDL/INSERT 실행 후 커밋, 실패 시 롤백
# 닫힌 연결에서 rollback이 또 실패해 원래 오류를 가리지 않도록 보강
# ============================================================
def exec_sql(sql, params=None, title=""):
    ensure_conn()
    t0 = time.time()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()
    except Exception:
        try:
            conn.rollback()
        except Exception:
            pass                       # 이미 닫힌 연결이면 무시하고 원래 오류를 그대로 전달
        raise
    print(f"✔ {title} 완료 ({time.time()-t0:,.0f}초)")

def show_activity():
    """현재 DB에서 실행 중인 다른 작업 확인 (대용량 작업이 겹치지 않게)"""
    df = query_df("""
        SELECT pid, state, now() - query_start AS running, left(query, 80) AS query
        FROM pg_stat_activity
        WHERE datname = current_database() AND pid <> pg_backend_pid()
          AND state <> 'idle'""")
    print("실행 중인 다른 작업:", "없음" if df.empty else "")
    if not df.empty:
        display(df)

show_activity()      # 다른 INSERT/CREATE 작업이 보이면 끝날 때까지 기다린 뒤 진행

실행 중인 다른 작업: 없음


###2-2.[DB저장] 자전거 종류 구분

* 대여이력을 이용하여 구분합니다.

* 일반자전거(성인) / 새싹자전거(어린이)
*	LCD자전거(QR 이전)

In [ ]:
# ============================================================
# [자전거] 원본 전체를 한 번 훑어 (자전거, 유형) 조합 + 건수/최초·최종 사용일 저장
#   결과는 자전거 수만큼의 작은 테이블 (이후 점검은 이 테이블에서 즉시 수행)
# ============================================================
exec_sql("DROP TABLE IF EXISTS bike_stage", title="기존 스테이징 삭제")

exec_sql("""
    CREATE TABLE bike_stage AS
    SELECT bike_no, bicycle_type,
           COUNT(*)         AS cnt,
           MIN(rental_date) AS first_dt,
           MAX(rental_date) AS last_dt
    FROM rental_history_raw
    GROUP BY bike_no, bicycle_type
""", title="bike_stage 생성 (전체 스캔, 수 분 소요)")

[재접속 완료]
✔ 기존 스테이징 삭제 완료 (1초)
✔ bike_stage 생성 (전체 스캔, 수 분 소요) 완료 (213초)


In [ ]:
# ============================================================
# [자전거] 한 자전거에 유형이 2개 이상이면 중단, 없으면 bike_info 테이블 생성
#      bike_stage에서 유형 충돌 점검 → 없으면 bike 테이블 생성 (작은 테이블이라 수 초)
# ============================================================
ensure_conn()
conflict = query_df("""
    SELECT bike_no, COUNT(*) AS n_type,
           string_agg(bicycle_type || '(' || cnt || ')', ', ' ORDER BY last_dt) AS types
    FROM bike_stage GROUP BY bike_no HAVING COUNT(*) > 1
    ORDER BY n_type DESC, bike_no LIMIT 50""")

total = query_df("SELECT COUNT(DISTINCT bike_no) AS bikes, COUNT(*) AS pairs FROM bike_stage")
print(total.to_string(index=False))

if not conflict.empty:
    print(f"\n⚠ 유형이 2개 이상인 자전거가 있습니다 (상위 50건). 처리 기준을 정한 뒤 진행하세요.")
    display(conflict)
else:
    exec_sql("DROP TABLE IF EXISTS bike_info", title="기존 bike 삭제")
    exec_sql("""
        CREATE TABLE bike_info (
            bike_no      TEXT PRIMARY KEY,     -- 자전거 아이디
            bicycle_type TEXT NOT NULL         -- 자전거 유형
        )""", title="bike 테이블 생성")
    exec_sql("""
        INSERT INTO bike_info (bike_no, bicycle_type)
        SELECT bike_no, bicycle_type FROM bike_stage ORDER BY bike_no
    """, title="bike 적재")

    display(query_df("SELECT bicycle_type, COUNT(*) AS bikes FROM bike_info GROUP BY 1 ORDER BY 2 DESC"))
    display(query_df("SELECT * FROM bike_info ORDER BY bike_no LIMIT 5"))

[재접속 완료]
 bikes  pairs
 51757  51757
✔ 기존 bike 삭제 완료 (1초)
✔ bike 테이블 생성 완료 (1초)
✔ bike 적재 완료 (1초)


,bicycle_type,bikes
0,일반자전거,44179
1,새싹자전거,5063
2,LCD자전거,2515


,bike_no,bicycle_type
0,SPB-00025,LCD자전거
1,SPB-00291,LCD자전거
2,SPB-00305,LCD자전거
3,SPB-00385,LCD자전거
4,SPB-00811,LCD자전거


# ↑여기까지 테이블은 생성됨.

# 대여이력 연도별 테이블 미생성

##확인

In [9]:
# ============================================================
# [상태 확인] 분리 결과 테이블이 이미 있는지, 비어 있는지, 인덱스가 있는지 (읽기 전용)
#
# DB툴에서 실행해서 확인
# SET maintenance_work_mem = '1GB';
# CREATE INDEX IF NOT EXISTS idx_raw_rental_date ON rental_history_raw (rental_date COLLATE "C");
# ANALYZE rental_history_raw;
#
# SELECT phase, blocks_done, blocks_total FROM pg_stat_progress_create_index;
# ============================================================
ensure_conn()
tbls = query_df("""
    SELECT c.relname AS table_name,
           CASE c.relkind WHEN 'p' THEN 'partitioned(부모)' ELSE 'table' END AS kind
    FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = current_schema() AND c.relkind IN ('r','p')
      AND (c.relname LIKE 'rental_history%' OR c.relname = 'rental_split_log')
    ORDER BY 1""")
display(tbls)

# 이미 있는 분리 결과 테이블이 비어 있는지 (원본/적재 로그 제외)
for t in tbls["table_name"]:
    if t in ("rental_history_raw", "rental_history_load_log"):
        continue
    has = query_df(f"SELECT EXISTS (SELECT 1 FROM {t}) AS has_rows")["has_rows"][0]
    print(f"{t:30s} → {'데이터 있음' if has else '비어 있음'}")

print("\n원본 인덱스:", "있음 ✔" if len(query_df("SELECT 1 FROM pg_indexes WHERE indexname = 'idx_raw_rental_date'")) else "없음 ✘")

,table_name,kind
0,rental_history_2021,table
1,rental_history_2022,table
2,rental_history_2023,table
3,rental_history_2024,table
4,rental_history_2025,table
5,rental_history_load_log,table
6,rental_history_no_return,table
7,rental_history_raw,table
8,rental_history_time_error,table
9,rental_split_log,table


rental_history_2021            → 데이터 있음
rental_history_2022            → 비어 있음
rental_history_2023            → 데이터 있음
rental_history_2024            → 비어 있음
rental_history_2025            → 비어 있음
rental_history_no_return       → 비어 있음
rental_history_time_error      → 비어 있음
rental_split_log               → 데이터 있음

원본 인덱스: 있음 ✔


In [16]:
# ============================================================
# [대여이력 분리 1] 연도별 테이블 5개 + 예외 테이블 2개 + 진행 로그 테이블 생성, SQL 정의
#   연도별    : 정상 이용 건 (타입 변환 적용, 17개 컬럼 모두 유지)
#   no_return : 반납일시가 없는 건 (원본 문자열 그대로 + 사유)
#   time_error: 대여일시 불량 또는 반납일시 < 대여일시인 건 (원본 문자열 그대로 + 사유)
# ============================================================
YEARS = [2021, 2022, 2023, 2024, 2025]

year_ddl = """
CREATE TABLE IF NOT EXISTS {tbl} (
    bike_no            text        NOT NULL,
    rental_date        timestamp   NOT NULL,
    rental_station_no  integer,
    rental_station_nm  text,
    rental_hold        integer,
    return_date        timestamp   NOT NULL,
    return_station_no  integer,
    return_station_nm  text,
    return_hold        integer,
    rental_time        integer,
    rental_distance    double precision,
    birth              integer,
    sex                text,
    user_type          text,
    rental_station_id  text,
    return_station_id  text,
    bicycle_type       text
)"""

# 분류 규칙 (서로 겹치지 않고 빠지는 행이 없음)
#   대여일시 형식 불량 → time_error / 반납일시 NULL·형식 불량 → no_return
#   반납일시 < 대여일시 → time_error / 그 외 → 연도별 테이블
# 날짜 비교는 COLLATE "C" (인덱스 사용 + 빠른 비교). 형식은 길이(19자)로 확인하고
# 그 외 깨진 값(예: 13월)은 변환 시 오류로 드러남
MAIN_SQL = r"""
INSERT INTO {tbl}
SELECT bike_no,
       rental_date::timestamp,
       NULLIF(rental_station_no,'\N')::int,  rental_station_nm,
       NULLIF(rental_hold,'\N')::int,
       return_date::timestamp,
       NULLIF(return_station_no,'\N')::int,  return_station_nm,
       NULLIF(return_hold,'\N')::int,
       NULLIF(rental_time,'\N')::int,
       NULLIF(rental_distance,'\N')::float8,
       NULLIF(birth,'\N')::int,
       NULLIF(sex,'\N'),
       user_type, rental_station_id, return_station_id, bicycle_type
FROM rental_history_raw
WHERE rental_date COLLATE "C" >= %s AND rental_date COLLATE "C" < %s
  AND length(rental_date) = 19 AND length(return_date) = 19
  AND return_date COLLATE "C" >= rental_date COLLATE "C"
"""

NO_RETURN_SQL = r"""
INSERT INTO rental_history_no_return
SELECT r.*, CASE WHEN r.return_date IS NULL THEN 'RETURN_NULL' ELSE 'RETURN_BAD_FORMAT' END
FROM rental_history_raw r
WHERE r.rental_date COLLATE "C" >= %s AND r.rental_date COLLATE "C" < %s
  AND length(r.rental_date) = 19
  AND (r.return_date IS NULL OR length(r.return_date) <> 19)
"""

TIME_ERROR_SQL = r"""
INSERT INTO rental_history_time_error
SELECT r.*, CASE WHEN length(r.rental_date) <> 19 THEN 'RENTAL_DATE_BAD' ELSE 'RETURN_BEFORE_RENTAL' END
FROM rental_history_raw r
WHERE r.rental_date COLLATE "C" >= %s AND r.rental_date COLLATE "C" < %s
  AND ( length(r.rental_date) <> 19
        OR (length(r.return_date) = 19 AND r.return_date COLLATE "C" < r.rental_date COLLATE "C") )
"""

ensure_conn()
with conn.cursor() as cur:
    for y in YEARS:
        cur.execute(year_ddl.replace("{tbl}", f"rental_history_{y}"))
    cur.execute("CREATE TABLE IF NOT EXISTS rental_history_no_return  (LIKE rental_history_raw, reason text)")
    cur.execute("CREATE TABLE IF NOT EXISTS rental_history_time_error (LIKE rental_history_raw, reason text)")
    cur.execute("""CREATE TABLE IF NOT EXISTS rental_split_log (
                       part_key text PRIMARY KEY,      -- 예: 2021-01-01:main / :no_return / :time_error
                       row_cnt  bigint,
                       done_at  timestamp DEFAULT now())""")
conn.commit()
print("테이블 준비 완료")

테이블 준비 완료


In [10]:
# ============================================================
# [대여이력 분리 2] 7일 단위 청크로 분리 (쿼리 1회가 짧아 연결 끊김에 강함)
#   - 청크는 연도 경계를 넘지 않음 (연도별 테이블로 가야 하므로)
#   - 청크마다 [정상 / 반납없음 / 시간오류] 3단계를 각각 한 트랜잭션 + 로그로 처리
#   - 끊겨도 같은 셀을 다시 실행하면 완료된 단계는 건너뛰고 이어서 진행
# ============================================================
def make_chunks(days=7):
    """(연도, 시작일, 종료일[미포함]) 목록 생성"""
    chunks = []
    for y in YEARS:
        d, y_end = pd.Timestamp(f"{y}-01-01"), pd.Timestamp(f"{y+1}-01-01")
        while d < y_end:
            e = min(d + pd.Timedelta(days=days), y_end)
            chunks.append((y, d.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d")))
            d = e
    return chunks

def fmt(n):
    return "-" if n is None else f"{n:,}"          # 이미 완료된 단계는 '-' 로 표시

def get_done_parts():
    return set(query_df("SELECT part_key FROM rental_split_log")["part_key"])

def show_activity():
    """현재 DB에서 실행 중인 다른 작업 확인 (대용량 작업이 겹치지 않게)"""
    df = query_df("""SELECT pid, state, now() - query_start AS running, left(query, 80) AS query
                     FROM pg_stat_activity
                     WHERE datname = current_database() AND pid <> pg_backend_pid() AND state <> 'idle'""")
    print("실행 중인 다른 작업:", "없음" if df.empty else "")
    if not df.empty:
        display(df)

def run_part(part_key, sql, params, retries=2):
    """SQL 실행 + 진행 로그 기록을 한 트랜잭션으로 처리. 연결이 끊기면 재접속 후 재시도"""
    for attempt in range(retries + 1):
        ensure_conn()
        if attempt > 0 and part_key in get_done_parts():     # 커밋 직후 끊겼던 경우 중복 방지
            return None
        try:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                n = cur.rowcount
                cur.execute("INSERT INTO rental_split_log (part_key, row_cnt) VALUES (%s, %s)", [part_key, n])
            conn.commit()
            return n
        except (psycopg2.OperationalError, psycopg2.InterfaceError) as e:
            print(f"   [연결 오류 {attempt+1}회차] {str(e).strip()[:80]}")
            try: conn.rollback()
            except Exception: pass
            if attempt == retries:
                raise
            time.sleep(5)
        except Exception:
            try: conn.rollback()
            except Exception: pass
            raise

def split_chunks(chunks, limit=None):
    done = get_done_parts()
    todo = [c for c in chunks if not {f"{c[1]}:{p}" for p in ("main", "no_return", "time_error")} <= done]
    if limit:
        todo = todo[:limit]
    print(f"전체 {len(chunks)}개 청크 / 이번 처리 {len(todo)}개")
    t_all = time.time()
    for i, (y, start, end) in enumerate(todo, 1):
        t0, res = time.time(), {}
        steps = [("main", MAIN_SQL.replace("{tbl}", f"rental_history_{y}")),
                 ("no_return", NO_RETURN_SQL), ("time_error", TIME_ERROR_SQL)]
        for name, sql in steps:
            key = f"{start}:{name}"
            res[name] = None if key in done else run_part(key, sql, [start, end])
        avg = (time.time() - t_all) / i
        print(f"[{i}/{len(todo)}] {start}~{end}  정상 {fmt(res['main'])} / 반납없음 {fmt(res['no_return'])}"
              f" / 시간오류 {fmt(res['time_error'])}  ({time.time()-t0:,.0f}초, 남은 예상 약 {avg*(len(todo)-i)/60:,.0f}분)")

In [17]:
# 사전 점검: 인덱스가 없으면 중단 (없으면 청크마다 전체 스캔이 되어 연결이 끊길 수 있음)
if not len(query_df("SELECT 1 FROM pg_indexes WHERE indexname = 'idx_raw_rental_date'")):
    raise RuntimeError("원본 인덱스가 없습니다. 단계 1(pgAdmin)을 먼저 실행하세요.")
show_activity()

chunks = make_chunks(days=7)
print(f"청크 수: {len(chunks)}개")

split_chunks(chunks, limit=1)        # ① 시험: 첫 7일만. 시간과 건수 확인
# split_chunks(chunks)               # ② 확인 후 전체 실행 (끊겨도 다시 실행하면 이어서 진행)

실행 중인 다른 작업: 없음
청크 수: 265개
전체 265개 청크 / 이번 처리 1개
[1/1] 2021-02-19~2021-02-26  정상 385,746 / 반납없음 0 / 시간오류 0  (13초, 남은 예상 약 0분)


In [18]:
# 가장 큰 구간(2023-10 중순) 청크 1개로 최악의 소요 시간을 측정
target = next(c for c in chunks if c[1] <= "2023-10-15" < c[2])
print("시험 청크:", target)
split_chunks([target])

시험 청크: (2023, '2023-10-15', '2023-10-22')
전체 1개 청크 / 이번 처리 0개


In [21]:
split_chunks(chunks)      # 완료된 청크는 건너뛰고 나머지 263개를 순서대로 처리

전체 265개 청크 / 이번 처리 186개
[1/186] 2022-06-25~2022-07-02  정상 694,905 / 반납없음 0 / 시간오류 0  (17초, 남은 예상 약 51분)
[2/186] 2022-07-02~2022-07-09  정상 956,949 / 반납없음 0 / 시간오류 0  (29초, 남은 예상 약 70분)
[3/186] 2022-07-09~2022-07-16  정상 900,511 / 반납없음 0 / 시간오류 0  (28초, 남은 예상 약 75분)
[4/186] 2022-07-16~2022-07-23  정상 985,702 / 반납없음 0 / 시간오류 0  (29초, 남은 예상 약 78분)
[5/186] 2022-07-23~2022-07-30  정상 963,434 / 반납없음 0 / 시간오류 0  (28초, 남은 예상 약 79분)
[6/186] 2022-07-30~2022-08-06  정상 699,815 / 반납없음 0 / 시간오류 0  (24초, 남은 예상 약 78분)
[7/186] 2022-08-06~2022-08-13  정상 590,095 / 반납없음 0 / 시간오류 0  (20초, 남은 예상 약 75분)
[8/186] 2022-08-13~2022-08-20  정상 812,277 / 반납없음 0 / 시간오류 0  (26초, 남은 예상 약 75분)
[9/186] 2022-08-20~2022-08-27  정상 1,032,532 / 반납없음 0 / 시간오류 0  (33초, 남은 예상 약 77분)
[10/186] 2022-08-27~2022-09-03  정상 990,989 / 반납없음 0 / 시간오류 0  (31초, 남은 예상 약 78분)
[11/186] 2022-09-03~2022-09-10  정상 844,617 / 반납없음 0 / 시간오류 0  (27초, 남은 예상 약 77분)
[12/186] 2022-09-10~2022-09-17  정상 1,031,856 / 반납없음 0 / 시간오류 0  (34초, 남은 예상 약 79분)
[13/186] 

In [22]:
# 검증
raw_total = int(query_df("SELECT COALESCE(SUM(row_cnt),0) AS n FROM rental_history_load_log WHERE status = 'DONE'")["n"][0])
by_part = query_df("""SELECT split_part(part_key, ':', 2) AS part, SUM(row_cnt) AS cnt, COUNT(*) AS chunks
                      FROM rental_split_log GROUP BY 1 ORDER BY 1""")
display(by_part)
split_total = int(by_part["cnt"].sum()) if len(by_part) else 0
print(f"원본 적재 행 수: {raw_total:,} / 분리 합계: {split_total:,} / 차이: {raw_total - split_total:,}")

display(query_df("SELECT reason, COUNT(*) AS cnt FROM rental_history_no_return  GROUP BY 1"))
display(query_df("SELECT reason, COUNT(*) AS cnt FROM rental_history_time_error GROUP BY 1"))

,part,cnt,chunks
0,main,199133391,265
1,no_return,0,265
2,time_error,74,265


원본 적재 행 수: 199,133,465 / 분리 합계: 199,133,465 / 차이: 0


,reason,cnt


,reason,cnt
0,RETURN_BEFORE_RENTAL,74


In [23]:
# 첫 청크 정합성: 원본의 같은 기간 건수와 분리 결과가 같은지, 값/타입이 정상인지
raw_cnt = query_df("""SELECT COUNT(*) AS n FROM rental_history_raw
                      WHERE rental_date COLLATE "C" >= '2021-01-01' AND rental_date COLLATE "C" < '2021-01-08'""")["n"][0]
done_cnt = query_df("SELECT COUNT(*) AS n FROM rental_history_2021")["n"][0]
print(f"원본(해당 기간): {raw_cnt:,} / 분리 결과: {done_cnt:,} → {'일치 ✔' if raw_cnt == done_cnt else '불일치 ⚠'}")

display(query_df("SELECT * FROM rental_history_2021 LIMIT 5"))
display(query_df("""SELECT column_name, data_type FROM information_schema.columns
                    WHERE table_name = 'rental_history_2021' ORDER BY ordinal_position"""))
display(query_df("""SELECT MIN(rental_date) AS min_rental, MAX(rental_date) AS max_rental,
                           COUNT(*) FILTER (WHERE birth IS NULL)       AS birth_null,
                           COUNT(*) FILTER (WHERE sex IS NULL)         AS sex_null,
                           COUNT(*) FILTER (WHERE return_hold IS NULL) AS return_hold_null
                    FROM rental_history_2021"""))

원본(해당 기간): 157,514 / 분리 결과: 32,053,322 → 불일치 ⚠


,bike_no,rental_date,rental_station_no,rental_station_nm,rental_hold,return_date,return_station_no,return_station_nm,return_hold,rental_time,rental_distance,birth,sex,user_type,rental_station_id,return_station_id,bicycle_type
0,SPB-32649,2021-01-01 00:00:01,2719,마곡문화공원,0,2021-01-01 00:15:26,2743,동신아파트 104동 앞,0,15,2098.59,1994,None,내국인,ST-2035,ST-2399,일반자전거
1,SPB-30216,2021-01-01 00:00:10,670,삼육서울병원 버스정류장,0,2021-01-01 00:16:58,1405,망우역 1번출구,0,16,3092.16,1974,M,내국인,ST-2270,ST-568,일반자전거
2,SPB-32263,2021-01-01 00:00:23,1634,당고개공원 대여소,0,2021-01-01 00:12:30,1691,노원정보도서관,0,12,1939.47,1993,M,내국인,ST-608,ST-1642,일반자전거
3,SPB-31575,2021-01-01 00:00:33,1977,천왕역 1번 출입구 앞,0,2021-01-01 00:06:03,1952,천왕연지타운2단지 앞,0,5,810.91,1976,None,내국인,ST-1239,ST-875,일반자전거
4,SPB-41749,2021-01-01 00:00:40,635,시조사 앞,0,2021-01-01 00:33:24,1320,LG베스트샵 종암점,0,32,3472.56,2004,M,내국인,ST-402,ST-597,일반자전거


,column_name,data_type
0,bike_no,text
1,rental_date,timestamp without time zone
2,rental_station_no,integer
3,rental_station_nm,text
4,rental_hold,integer
5,return_date,timestamp without time zone
6,return_station_no,integer
7,return_station_nm,text
8,return_hold,integer
9,rental_time,integer


,min_rental,max_rental,birth_null,sex_null,return_hold_null
0,2021-01-01 00:00:01,2021-12-31 23:59:48,3322650,13098215,132005
